# WOOF — Evaluación del Modelo (sin re-entrenar)
**Carga el modelo desde un Kaggle Dataset y genera:**
- Matriz de confusión (binaria + top 20 razas)
- Reporte de precisión por clase (rango máx / mín)
- Coeficiente de determinación (Pseudo-R² de McFadden)

**Inputs necesarios en este notebook:**
1. `jessicali9530/stanford-dogs-dataset`
2. `alessiocorrado99/animals10`
3. `sergiodelcarpio/perros-vs-gatos`
4. Tu dataset con el modelo `.h5` (ej: `tu-usuario/woof-model-v3`)

In [ ]:
import os, json, numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import r2_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Configuración ────────────────────────────────────────────
IMG_SIZE   = 380
BATCH_SIZE = 32
SEED       = 42
VAL_SPLIT  = 0.2
MAX_NEG    = 2500

# Rutas datasets (mismo formato que el notebook de entrenamiento)
STANFORD_PATH  = '/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images'
ANIMALS10_PATH = '/kaggle/input/datasets/alessiocorrado99/animals10/raw-img'
DOGS_CATS_BASE = '/kaggle/input/datasets/sergiodelcarpio/perros-vs-gatos/gatos_perros/training_set'

# ── Buscar el .h5 automáticamente en /kaggle/input/ ──────────
MODEL_H5 = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.h5'):
            MODEL_H5 = os.path.join(root, f)
            break
    if MODEL_H5:
        break

if MODEL_H5 is None:
    raise FileNotFoundError('No se encontró ningún .h5 en /kaggle/input/. '
                            'Agrega tu dataset del modelo como input.')

print(f'Modelo encontrado: {MODEL_H5}')
print(f'Tamaño: {os.path.getsize(MODEL_H5)/1024**2:.1f} MB')

# Verificar que los datasets existen
for nombre, ruta in [('Stanford Dogs', STANFORD_PATH),
                      ('Animals10',     ANIMALS10_PATH),
                      ('Perros vs Gatos', DOGS_CATS_BASE)]:
    existe = os.path.exists(ruta)
    print(f'  {nombre}: {"OK" if existe else "NO ENCONTRADO"}  →  {ruta}')


In [ ]:
# ── Cargar modelo ────────────────────────────────────────────
print('Cargando modelo...')
model = load_model(MODEL_H5)
print('Modelo cargado.')
print(f'Parámetros: {model.count_params():,}')
print(f'Clases de salida: {model.output_shape[-1]}')

In [ ]:
# ── Reconstruir DataFrame con el mismo split del entrenamiento ─
razas = sorted(os.listdir(STANFORD_PATH))
rows_s = []
for r in razas:
    breed = r.split('-', 1)[-1].lower()
    for f in os.listdir(os.path.join(STANFORD_PATH, r)):
        rows_s.append({'filepath': os.path.join(STANFORD_PATH, r, f), 'label': breed})
df_stanford = pd.DataFrame(rows_s)

# Buscar carpeta dogs/
DOGS_DIR = CATS_DIR = None
for c in [f'{DOGS_CATS_BASE}/gatos_perros/training_set',
          f'{DOGS_CATS_BASE}/training_set', DOGS_CATS_BASE]:
    if os.path.exists(os.path.join(c, 'dogs')):
        DOGS_DIR = os.path.join(c, 'dogs')
        CATS_DIR = os.path.join(c, 'cats')
        break

rows_pvsg = [{'filepath': os.path.join(DOGS_DIR, f), 'label': 'dog_generic'}
             for f in os.listdir(DOGS_DIR) if os.path.isfile(os.path.join(DOGS_DIR, f))]

cane_dir = os.path.join(ANIMALS10_PATH, 'cane')
rows_a10d = [{'filepath': os.path.join(cane_dir, f), 'label': 'dog_generic'}
             for f in os.listdir(cane_dir) if os.path.isfile(os.path.join(cane_dir, f))]

rng = np.random.RandomState(SEED)
rows_neg = []
for cls in sorted(os.listdir(ANIMALS10_PATH)):
    if cls.lower() == 'cane': continue
    d = os.path.join(ANIMALS10_PATH, cls)
    if not os.path.isdir(d): continue
    files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
    if len(files) > MAX_NEG:
        files = rng.choice(files, MAX_NEG, replace=False).tolist()
    rows_neg += [{'filepath': os.path.join(d, f), 'label': 'not_a_dog'} for f in files]

rows_cats = [{'filepath': os.path.join(CATS_DIR, f), 'label': 'not_a_dog'}
             for f in os.listdir(CATS_DIR) if os.path.isfile(os.path.join(CATS_DIR, f))]

df_all = pd.concat([
    df_stanford,
    pd.DataFrame(rows_pvsg),
    pd.DataFrame(rows_a10d),
    pd.DataFrame(rows_neg),
    pd.DataFrame(rows_cats)
], ignore_index=True)

_, df_val = train_test_split(df_all, test_size=VAL_SPLIT,
                              stratify=df_all['label'], random_state=SEED)
df_val = df_val.reset_index(drop=True)
print(f'Total dataset: {len(df_all):,}  |  Validación: {len(df_val):,}')

In [ ]:
# ── Generador de validación ───────────────────────────────────
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_gen = val_datagen.flow_from_dataframe(
    dataframe=df_val, x_col='filepath', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
idx_to_class    = {v: k for k, v in val_gen.class_indices.items()}
not_dog_idx     = val_gen.class_indices.get('not_a_dog', -1)
steps_val       = len(df_val) // BATCH_SIZE
print(f'Clases: {len(val_gen.class_indices)}  |  not_a_dog idx: {not_dog_idx}')

In [ ]:
# ── Recolectar predicciones (~10 min) ─────────────────────────
val_gen.reset()
y_true_all, y_pred_all, y_probs_all = [], [], []

for _ in tqdm(range(steps_val + 1), desc='Prediciendo val set'):
    try:
        X_b, y_b = next(val_gen)
    except StopIteration:
        break
    probs = model.predict(X_b, verbose=0)
    y_true_all.extend(np.argmax(y_b, axis=1))
    y_pred_all.extend(np.argmax(probs, axis=1))
    y_probs_all.extend(probs)

n = min(len(y_true_all), len(df_val))
y_true_all  = np.array(y_true_all[:n])
y_pred_all  = np.array(y_pred_all[:n])
y_probs_all = np.array(y_probs_all[:n])

acc_val = np.mean(y_true_all == y_pred_all)
print(f'Muestras: {n:,}  |  Accuracy calculada: {acc_val:.2%}')

## 1. Matriz de Confusión — Binaria (Perro / No es Perro)

In [ ]:
y_true_bin = (y_true_all == not_dog_idx).astype(int)
y_pred_bin = (y_pred_all == not_dog_idx).astype(int)

cm_bin = confusion_matrix(y_true_bin, y_pred_bin)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_bin, display_labels=['Perro', 'No es perro']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Matriz de Confusión — Binaria (Perro / No es Perro)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

TP, FN = cm_bin[0, 0], cm_bin[0, 1]
FP, TN = cm_bin[1, 0], cm_bin[1, 1]
print(f'Verdaderos Positivos (perros correctos):  {TP:>6,}')
print(f'Falsos Negativos (perros perdidos):       {FN:>6,}')
print(f'Falsos Positivos (no-perros como perro):  {FP:>6,}')
print(f'Verdaderos Negativos (no-perros ok):      {TN:>6,}')
print(f'\nSensibilidad (recall perros):  {TP/(TP+FN):.2%}')
print(f'Especificidad (recall no-dog): {TN/(TN+FP):.2%}')

## 2. Matriz de Confusión — Top 20 Razas más Confundidas

In [ ]:
mask_dog  = (y_true_all != not_dog_idx) & (y_pred_all != not_dog_idx)
y_true_dog = y_true_all[mask_dog]
y_pred_dog = y_pred_all[mask_dog]

cm_full = confusion_matrix(y_true_dog, y_pred_dog)
errors  = cm_full.sum(axis=1) - np.diag(cm_full)
top20   = np.argsort(errors)[::-1][:20]

# Mapear índices de subconjunto a nombres
unique_dog_classes = sorted(np.unique(y_true_dog))
idx_map = {i: unique_dog_classes[i] for i in range(len(unique_dog_classes))}
labels_top20 = [idx_to_class[idx_map[i]] for i in top20]

cm_top20 = cm_full[np.ix_(top20, top20)]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_top20, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=labels_top20, yticklabels=labels_top20,
            linewidths=0.4, ax=ax)
ax.set_title('Matriz de Confusión — Top 20 Razas con más Errores', fontsize=13, pad=12)
ax.set_xlabel('Predicción', fontsize=11)
ax.set_ylabel('Real', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 3. Rango de Precisión por Clase

In [ ]:
classes_present = sorted(np.unique(y_true_all))
names_present   = [idx_to_class[i] for i in classes_present]

report = classification_report(
    y_true_all, y_pred_all,
    labels=classes_present, target_names=names_present,
    output_dict=True, zero_division=0
)
df_rep = pd.DataFrame(report).T
df_rep = df_rep.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
df_rep = df_rep[['precision', 'recall', 'f1-score', 'support']].astype(
    {'precision': float, 'recall': float, 'f1-score': float, 'support': int}
)

df_sorted  = df_rep.sort_values('precision', ascending=False)
prec_valid = df_rep.loc[df_rep['support'] >= 5, 'precision']

print('='*65)
print('  RANGO DE PRECISIÓN POR CLASE')
print('='*65)
print('\n  TOP 5 — Mayor precisión:')
print(f'  {"Clase":<35} {"Precisión":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-'*60)
for name, row in df_sorted.head(5).iterrows():
    print(f'  {name:<35} {row["precision"]:>9.1%} {row["recall"]:>7.1%} {row["f1-score"]:>7.1%}')

print('\n  BOTTOM 5 — Menor precisión:')
print(f'  {"Clase":<35} {"Precisión":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-'*60)
for name, row in df_sorted.tail(5).iterrows():
    print(f'  {name:<35} {row["precision"]:>9.1%} {row["recall"]:>7.1%} {row["f1-score"]:>7.1%}')

print(f'\n  Precisión MÁXIMA  : {prec_valid.max():.2%}  →  {prec_valid.idxmax()}')
print(f'  Precisión MÍNIMA  : {prec_valid.min():.2%}  →  {prec_valid.idxmin()}')
print(f'  Precisión PROMEDIO: {prec_valid.mean():.2%}')
print(f'  Precisión MEDIANA : {prec_valid.median():.2%}')
print('='*65)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(prec_valid, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(prec_valid.mean(),   color='red',    ls='--', lw=1.5, label=f'Media  {prec_valid.mean():.1%}')
ax.axvline(prec_valid.median(), color='orange', ls='--', lw=1.5, label=f'Mediana {prec_valid.median():.1%}')
ax.set_title('Distribución de Precisión por Clase (122 clases)', fontsize=13)
ax.set_xlabel('Precisión')
ax.set_ylabel('Número de clases')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Coeficiente de Determinación (R²) — Pseudo-R² de McFadden

In [ ]:
NUM_C      = y_probs_all.shape[1]
true_probs = np.clip(y_probs_all[np.arange(n), y_true_all], 1e-9, 1.0)

LL_model    = np.sum(np.log(true_probs))
LL_null     = n * np.log(1.0 / NUM_C)
r2_mcfadden = 1.0 - (LL_model / LL_null)

classes_all = sorted(val_gen.class_indices.values())
y_onehot    = label_binarize(y_true_all, classes=classes_all)
r2_efron    = r2_score(y_onehot, y_probs_all, multioutput='uniform_average')

if   r2_mcfadden >= 0.40: nivel = 'EXCELENTE'
elif r2_mcfadden >= 0.20: nivel = 'BUENO'
elif r2_mcfadden >= 0.10: nivel = 'ACEPTABLE'
else:                      nivel = 'DÉBIL'

print('='*55)
print('  COEFICIENTE DE DETERMINACIÓN (R²)')
print('='*55)
print(f'  Pseudo-R² de McFadden : {r2_mcfadden:.4f}  ({r2_mcfadden:.2%})')
print(f'  R² de Efron (probs)   : {r2_efron:.4f}  ({r2_efron:.2%})')
print(f'\n  Interpretación:')
print( '    > 0.40  →  Excelente')
print( '    0.20-0.40  →  Bueno')
print( '    0.10-0.20  →  Aceptable')
print(f'\n  Nuestro modelo: {nivel}  (R²={r2_mcfadden:.4f})')
print('='*55)

metricas = {
    'Top-1 Accuracy': acc_val,
    'Pseudo-R² (McFadden)': r2_mcfadden,
    'R² (Efron)': r2_efron,
}
fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(list(metricas.keys()), list(metricas.values()),
               color=['#2196F3', '#FF9800', '#9C27B0'])
for bar, val in zip(bars, metricas.values()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_xlim(0, 1.15)
ax.set_title('Resumen de Métricas — EfficientNetB4', fontsize=13)
ax.axvline(1.0, color='gray', ls='--', lw=1, alpha=0.5)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()